In [ ]:

from pymongo import MongoClient
import pandas as pd

try: 
    client = MongoClient("localhost", 27017)
    print("Connected successfully!!!") 
except:
    print("Could not connect to MongoDB")

db = client["flask_db"]
activity = db.activity



In [ ]:
query = {"project": "656fadd102ae94a7686aae62", "editingLines": {"$exists": True, "$ne": None}}

cursor = activity.find(query)

df = pd.DataFrame(list(cursor))
#df = df.astype({"text": str, "state": str, "line": str, "username": str, "project": str, "file": str, "message": str})

col_names = df.columns.tolist()
dtypes = df.dtypes
df.head()

In [ ]:
print("column names", col_names)
print("num rows", len(df))

print("average len", df["editingLines"].apply(len).mean())
print("median len", df["editingLines"].apply(len).median())
print("mode len", df["editingLines"].apply(len).mode().tolist())

In [ ]:
print("unique files in project", df["file"].unique())

In [ ]:
df1 = df["username"].value_counts().sort_values()
df["line_counts"] = df["editingLines"].apply(len)

df2 = df.groupby(["username"])["line_counts"].sum().sort_values()
print(df.head())

print(df1)
print(df2)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import colorcet as cc

sns.set_theme(style="ticks")

print(df["username"].unique())

fig, (ax1, ax2) = plt.subplots(1, 2, constrained_layout = True) 

ax1 = sns.barplot(df1, ax=ax1)
ax1.set(xlabel='Username', ylabel='Number of Edits')
ax1.set_xticklabels(ax1.get_xticklabels(), rotation=40, ha="right")


ax2 = sns.barplot(df2, ax=ax2)
ax2.set(xlabel="Username", ylabel="Number of lines edited")
ax2.set_xticklabels(ax2.get_xticklabels(), rotation=40, ha="right")




In [ ]:
df["edits"] = df["editingLines"].apply(len)
df["cum_edits"] = df["edits"].cumsum()

fig, ax = plt.subplots(1, 1, constrained_layout = True) 

ax = sns.lineplot(df["cum_edits"], ax=ax)
ax.set(xlabel='Edit Number', ylabel='Lines Edited')
ax.set_xticklabels(ax.get_xticklabels(), rotation=40, ha="right")


# File edits across time

Currenly only showing .tex or .bib files

In [ ]:


pivot_df = pd.pivot_table(df, values="edits", index="timestamp", columns="file", aggfunc="sum", fill_value=0)

relevant_files = [col for col in pivot_df.columns if '.tex' in col or '.bib' in col]

pivot_df = pivot_df[relevant_files]
window_size = 50  # Adjust the window size as needed
pivot_df = pivot_df.rolling(window=window_size).mean()

print(pivot_df.index)
pivot_df.index = pd.to_datetime(pivot_df.index, unit="ms")

fig, ax = plt.subplots(figsize=(12, 6)) 

#colors = sns.color_palette('hls', len(pivot_df.columns))
colors = sns.color_palette(cc.glasbey, n_colors=len(pivot_df.columns))

ax.set_prop_cycle('color', colors)

ax.stackplot(pivot_df.index, pivot_df.values.T, labels=pivot_df.columns, alpha=0.7, baseline="sym")

# Add labels and title
ax.set_xlabel('Timestamp')
ax.set_ylabel('Lines edited')
ax.set_title('Edits by file')
ax.set_ylim((-50,50))
ax.axhline(0, color="black", ls="--");
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))


ax.legend(pivot_df.columns,loc='upper center', bbox_to_anchor=(0.5, -0.05),
          fancybox=True, shadow=True, ncol=5)



In [ ]:
import diff_match_patch as dmp_module

dmp = dmp_module.diff_match_patch()

In [ ]:
data = df["state"].value_counts()
print(data)
sns.barplot(data)

In [ ]:
df.head()

idx = 21896

print(df.iloc[idx]["state"])
print(df.iloc[idx]["file"])
print(df.iloc[idx]["text"])
print(df.iloc[idx]["revision"])
print("\nLen:", len(df.iloc[idx]["revision"]))

In [ ]:
import re

def get_section_edited(text, line, editingLines):
    edit_line = int(line) - int(editingLines[0])

    matches = list(re.finditer("\\\\section", text))

    if not matches:
        return None

    end = matches[-1].start()
    # -1 so a failed 'rfind' maps to the first line.
    newline_table = {-1: 0}
    for i, m in enumerate(re.finditer('\\n', text), 1):
        # Don't find newlines past our last match.
        offset = m.start()
        if offset > end:
            break
        newline_table[offset] = i

    matches_idx = []
    
    for m in matches:
        newline_offset = text.rfind('\n', 0, m.start())
        newline_end = text.find('\n', m.end())  # '-1' gracefully uses the end.
        line = text[newline_offset + 1:newline_end]
        line_number = newline_table[newline_offset]
        #print (line_number, line)

        matches_idx.append((line_number, line))

    if (len(matches_idx) == 0):
        #print("no matches", matches_idx)
        return None
    elif (edit_line < matches_idx[0][0]):
        return "no section"
        #raise Exception("too early")
    else:
        #print("edit_line", edit_line, "output:", next(((line_number, line) for (line_number, line) in matches_idx if line_number < edit_line), 0))
        string = next((line for (line_number, line) in matches_idx if line_number < edit_line), 0)
        if (isinstance(string, str)):
            return string[string.find("{")+1:string.find("}")]
        else:
            return None
        

    

In [ ]:

df1 = df.loc[df["file"] == "main.tex"]


idx = 2

state = df1.iloc[idx]["state"]
fname = df1.iloc[idx]["file"]
text = df1.iloc[idx]["text"]
changes = df1.iloc[idx]["changes"]
revision = df1.iloc[idx]["revision"]
editingLines = df1.iloc[idx]["editingLines"]
line = df1.iloc[idx]["line"]



print("length", len(text))
print("total lines", text.count("\n") + 1)
print("also total lines", len(editingLines))
print("editLine", editLine)
print("absolute line", line)

print("\n\n\n")
get_section_edited(text, line, editingLines)

In [ ]:
df1["section"] = df1.apply(lambda x: get_section_edited(x["text"], x["line"], x["editingLines"]), axis=1)

fig, ax = plt.subplots(figsize=(12, 6)) 

sns.barplot(df1["section"].value_counts(), ax=ax)
ax.set(xlabel='Section', ylabel='Edits')
ax.set_xticklabels(ax.get_xticklabels(), rotation=40, ha="right")

In [ ]:
pivot_df = pd.pivot_table(df1, index="timestamp", columns="section", aggfunc="size", fill_value=0)
print(pivot_df.head())

window_size = 3  # Adjust the window size as needed
pivot_df = pivot_df.rolling(window=window_size).mean()

print(pivot_df.index)
pivot_df.index = pd.to_datetime(pivot_df.index, unit="ms")

fig, ax = plt.subplots(figsize=(12, 6)) 

#colors = sns.color_palette('hls', len(pivot_df.columns))
colors = sns.color_palette(cc.glasbey, n_colors=len(pivot_df.columns))

ax.set_prop_cycle('color', colors)

ax.stackplot(pivot_df.index, pivot_df.values.T, labels=pivot_df.columns, alpha=0.7, baseline="sym")

# Add labels and title
ax.set_xlabel('Timestamp')
ax.set_ylabel('Lines edited')
ax.set_title('Edits by file')
ax.set_ylim((-0.75,0.75))
ax.axhline(0, color="black", ls="--");
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))


ax.legend(pivot_df.columns,loc='upper center', bbox_to_anchor=(0.5, -0.05),
          fancybox=True, shadow=True, ncol=5)